# Collaboration Patterns By Year (International, Domestic, Internal)

Using the count capability of the API, Dimensions allows you to quickly identify international, domestic, and inernal Collaboration

This notebook shows how to quickly identify international, domestic, and internal collaboration using the [Organizations data source](https://docs.dimensions.ai/dsl/datasource-organizations.html) and the [Publications data source](https://docs.dimensions.ai/dsl/datasource-publications.html) available via the [Dimensions Analytics API](https://docs.dimensions.ai/dsl/). 


In [1]:
import datetime
print("==\nCHANGELOG\nThis notebook was last run on %s\n==" % datetime.date.today().strftime('%b %d, %Y'))

==
CHANGELOG
This notebook was last run on Sep 10, 2025
==


## Prerequisites

This notebook assumes you have installed the [Dimcli](https://pypi.org/project/dimcli/) library and are familiar with the ['Getting Started' tutorial](https://api-lab.dimensions.ai/cookbooks/1-getting-started/1-Using-the-Dimcli-library-to-query-the-API.html).

In [2]:
!pip install dimcli plotly -U --quiet 

#
# load libraries
import dimcli
from dimcli.utils import *

import json, sys, time
import pandas as pd
import plotly.express as px  # plotly>=4.8.1
if not 'google.colab' in sys.modules:
  # make js dependecies local / needed by html exports
  from plotly.offline import init_notebook_mode
  init_notebook_mode(connected=True)

print("==\nLogging in..")
# https://digital-science.github.io/dimcli/getting-started.html#authentication
ENDPOINT = "https://app.dimensions.ai"
if 'google.colab' in sys.modules:
  import getpass
  KEY = getpass.getpass(prompt='API Key: ')  
  dimcli.login(key=KEY, endpoint=ENDPOINT)
else:
  KEY = ""
  dimcli.login(key=KEY, endpoint=ENDPOINT)
dsl = dimcli.Dsl()

Searching config file credentials for 'https://app.dimensions.ai' endpoint..


==
Logging in..
Dimcli - Dimensions API Client (v1.4)
Connected to: <https://app.dimensions.ai/api/dsl> - DSL v2.12
Method: dsl.ini file


## 1. Lookup the University that you are interested in

In [3]:
dsl.query("""
search organizations for "melbourne" return organizations
""").as_dataframe()

Returned Organizations: 20 (total = 23)
Time: 0.53s


,id,name,city_name,country_code,country_name,types,state_name,latitude,linkout,longitude,acronym
0,grid.772384.d,Trelleborg Marine Systems Melbourne Pty Ltd,Victoria,AU,Australia,[Company],NaN,NaN,NaN,NaN,NaN
1,grid.746611.3,Noyes Bros Melbourne Pty Ltd,NaN,AU,Australia,[Other],NaN,NaN,NaN,NaN,NaN
2,grid.631568.f,CityLink Melbourne Ltd,NaN,AU,Australia,[Other],NaN,NaN,NaN,NaN,NaN
3,grid.530408.a,Melbourne Institute of Technology,Melbourne,AU,Australia,[Nonprofit],Victoria,NaN,NaN,NaN,NaN
4,grid.511296.8,Melbourne Genomics Health Alliance,Melbourne,AU,Australia,[Nonprofit],Victoria,-37.797960,[https://www.melbournegenomics.org.au/],144.953870,NaN
5,grid.493437.e,RMIT Europe,Barcelona,ES,Spain,[Education],NaN,41.402576,[https://www.rmit.eu],2.194333,RMIT
6,grid.490309.7,Melbourne Sexual Health Centre,Carlton,AU,Australia,[Healthcare],Victoria,-37.803123,[https://www.mshc.org.au/],144.963840,MSHC
7,grid.477970.a,Melbourne Clinic,Richmond,AU,Australia,[Healthcare],Victoria,-37.815063,[http://www.themelbourneclinic.com.au/],144.999650,NaN
8,grid.474755.0,Leica Biosystems Melbourne Pty Ltd,Mt. Waverley,AU,Australia,[Company],NaN,NaN,[http://www.danaher.com/],NaN,NaN
9,grid.469061.c,Ridley College,Melbourne,AU,Australia,[Education],Victoria,-37.783780,[https://www.ridley.edu.au/],144.957660,NaN


In [4]:
institution = "grid.1008.9"

## 2. Publications output by year

In [15]:
allpubs = dsl.query(f"""
    
        search publications 
            where research_orgs.id = "{institution}"
            and type="article"
            and year > 2010
        return year
    
    
    """).as_dataframe()

allpubs.columns = ['year', 'pubs']
px.bar(allpubs, x="year", y="pubs")

Returned Year: 16
Time: 0.55s


## 3. International publications

In [16]:
international = dsl.query(f"""
    
        search publications 
            where research_orgs.id = "{institution}"
            and type="article"
            and count(research_org_countries) > 1
                 and year > 2010
        return year
    
    
    """).as_dataframe()

international.columns = ['year','international_count']
px.bar(international, x="year", y="international_count")

Returned Year: 16
Time: 0.58s


## 4. Domestic

In [17]:
domestic = dsl.query(f"""
    
        search publications 
            where research_orgs.id = "{institution}"
            and type="article"
            and count(research_org_countries) = 1
                 and year > 2010
        return year
    
    
    """).as_dataframe()

domestic.columns = ['year','domestic_count']
px.bar(domestic, x="year", y="domestic_count")

Returned Year: 16
Time: 0.68s


## 5. Internal

In [18]:
internal = dsl.query(f"""
    
        search publications 
            where research_orgs.id = "{institution}"
            and type="article"
            and count(research_orgs) = 1
                 and year > 2010
        return year
    
    
    """).as_dataframe()

internal.columns = ['year','internal_count']
px.bar(internal, x="year", y="internal_count")

Returned Year: 16
Time: 0.63s


## 6. Joining up All metrics together 

In [22]:
jdf = allpubs.set_index('year'). \
      merge(international, how='left', on='year'). \
      merge(domestic, how='left', on='year'). \
      merge(internal, how='left', on='year')

jdf

,year,pubs,international_count,domestic_count,internal_count
0,2021,19702,10330,9372,2920
1,2024,19104,10846,8258,2404
2,2023,18827,10338,8489,2641
3,2022,18677,10200,8477,2552
4,2020,18657,9649,9008,2782
5,2019,16617,8557,8060,2612
6,2018,15865,7934,7931,2538
7,2017,14873,7194,7679,2681
8,2016,13934,6563,7371,2608
9,2025,13886,8083,5803,1666


In [23]:
px.bar(jdf, title="University of Melbourne: publications collaboration")

## 7. How does this compare to Australia?

In [25]:
auallpubs = dsl.query("""
    
        search publications 
            where research_org_countries.name= "Australia"
            and type="article"
            and year > 2010
        return year
        
    """).as_dataframe()

auallpubs.columns = ['year', 'all_count']

auintpubs = dsl.query("""
    
        search publications 
            where research_org_countries.name= "Australia"
            and type="article"
            and year > 2010
            and count(research_org_countries) > 1
        return year
        
    """).as_dataframe()

auintpubs.columns = ['year', 'all_int_count']


audompubs = dsl.query("""
    
        search publications 
            where research_org_countries.name= "Australia"
            and type="article"
            and year > 2010
            and count(research_org_countries) = 1
        return year
        
    """).as_dataframe()

audompubs.columns = ['year', 'all_dom_count']

auinternalpubs = dsl.query("""
    
        search publications 
            where 
            research_org_countries.name= "Australia"
            and count(research_orgs) = 1
            and type="article"
            and year > 2010
        return year
        
    """).as_dataframe()

auinternalpubs.columns = ['year', 'all_internal_count']

audf = auallpubs.set_index('year'). \
      merge(auintpubs, how='left', on='year'). \
      merge(audompubs, how='left', on='year'). \
      merge(auinternalpubs, how='left', on='year'). \
      sort_values(by=['year'])

px.bar(audf, title="Australia: publications collaboration")

Returned Year: 16
Time: 0.63s
Returned Year: 16
Time: 0.52s
Returned Year: 16
Time: 0.48s
Returned Year: 16
Time: 5.60s


## 8. How does this compare to a different Institution  (University of Toronto)?

In [27]:
institution = "grid.17063.33"

allpubs = dsl.query(f"""
    
        search publications 
            where research_orgs.id = "{institution}"
            and type="article"
            and year > 2010
        return year
    
    
    """).as_dataframe()

allpubs.columns = ['year', 'pubs']



international = dsl.query(f"""
    
        search publications 
            where research_orgs.id = "{institution}"
            and type="article"
            and count(research_org_countries) > 1
                 and year > 2010
        return year
    
    
    """).as_dataframe()

international.columns = ['year', 'international_count']


domestic = dsl.query(f"""
    
        search publications 
            where research_orgs.id = "{institution}"
            and type="article"
            and count(research_org_countries) = 1
                 and year > 2010
        return year
    
    
    """).as_dataframe()

domestic.columns = ['year', 'domestic_count']

internal = dsl.query(f"""
    
        search publications 
            where research_orgs.id = "{institution}"
            and type="article"
            and count(research_orgs) = 1
                 and year > 2010
        return year
    
    
    """).as_dataframe()

internal.columns = ['year', 'internal_count']


jdf = allpubs.set_index('year'). \
      merge(international, how='left', on='year'). \
      merge(domestic, how='left', on='year'). \
      merge(internal, how='left', on='year')

px.bar(jdf, title="Univ. of Toronto: publications collaboration")


Returned Year: 16
Time: 0.49s
Returned Year: 16
Time: 6.01s
Returned Year: 16
Time: 0.63s
Returned Year: 16
Time: 0.49s


---
## Want to learn more?

Check out the [Dimensions API Lab](https://api-lab.dimensions.ai/) website, which contains many tutorials and reusable Jupyter notebooks for scholarly data analytics. 